# StegoCNN v4 — Arquitectura Multirrama con Atención SE

**Objetivo:** Clasificación multiclase (5 clases) de imágenes en `Cover` y 4 algoritmos de esteganografía / cifrado, usando:

- **Rama Espacial** — entrada `(256, 256, 15)` con residuos SRM (5 kernels × 3 canales RGB).
- **Rama Espectral** — entrada `(256, 256, 1)` con espectro DCT por bloques 8×8 del canal Y.
- **Bloques residuales** con `BatchNormalization` + activación **Mish** en cada rama.
- **Fusión tardía** vía `GlobalAveragePooling2D` + `Concatenate`.
- **Atención Squeeze-and-Excitation (SE)** después de la fusión, para que la red aprenda
  dinámicamente a pesar más el canal SRM o el DCT según la imagen.
- **Cabezal denso** fuertemente regularizado (Dropout 0.45 + L2).
- **AdamW** con `CosineDecay` del learning rate.
- **Class weights** automáticos (sklearn) + **Macro F1-Score** y matriz de confusión como métricas principales.

**Regla de oro:** ni `cv2.resize` con interpolación bicúbica/lanczos ni recompresión JPEG.
Usamos **Random Crops 256×256** sobre la imagen original como única forma de augmentation.


## 1. Imports y configuración global

In [2]:
import os, math, json, random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.ndimage import convolve
from scipy.fft import dctn
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers


In [3]:
# ── Reproducibilidad ─────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [4]:
print(f"TensorFlow: {tf.__version__}")
print(f"GPUs detectadas: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.21.0
GPUs detectadas: []


In [5]:
# Configuración
CROP_SIZE = 256 # Tamaño del recorte aleatorio (NO resize)
N_CLASSES = 5
BATCH_SIZE = 16
EPOCHS = 80
LR_INIT = 5e-4
WEIGHT_DECAY = 5e-5
L2_REG = 1e-4
DROPOUT = 0.45

In [6]:
# Rutas del dataset
ROOT_TRAIN = Path('dataset/train')
ROOT_VAL = Path('dataset/val')
ROOT_TEST = Path('dataset/test')

In [7]:
# Clases a clasficar
CLASES = ["original", "lsb", "bpcs", "dct", "pvd"]
NOMBRES_CLASES = CLASES

## 2. Extracción de características

Dos transformaciones complementarias que **NO destruyen** la señal estego:

- **SRM (Spatial Rich Model):** convolucionamos cada canal RGB con 5 filtros pasa-altos.
  Resultado: `(H, W, 15)` con el ruido residual (que es donde vive la firma del LSB clásico).
- **DCT por bloques 8×8:** sobre el canal Y (luminancia), capturamos la energía
  en frecuencias medias/altas — sensible a los algoritmos que escriben directamente en el dominio frecuencial (F5, JSteg).


In [8]:
# KERNELS SRM (rama espacial)
KERNELS_SRM = [
    np.array([[ 0,  1,  0],
              [ 1, -4,  1],
              [ 0,  1,  0]], dtype=np.float32) / 4.0,     # Laplaciano 4-conectado
    np.array([[ 1,  1,  1],
              [ 1, -8,  1],
              [ 1,  1,  1]], dtype=np.float32) / 8.0,     # Laplaciano 8-conectado
    np.array([[ 0,  0,  0],
              [ 1, -2,  1],
              [ 0,  0,  0]], dtype=np.float32) / 2.0,     # 2da derivada horizontal
    np.array([[ 0,  1,  0],
              [ 0, -2,  0],
              [ 0,  1,  0]], dtype=np.float32) / 2.0,     # 2da derivada vertical
    np.array([[ 1,  0,  1],
              [ 0, -4,  0],
              [ 1,  0,  1]], dtype=np.float32) / 4.0,     # 2da derivada diagonal
]

def calcular_residuos_srm(img_array: np.ndarray, T: float = 4.0) -> np.ndarray:
    """
    Convoluciona cada canal RGB con los 5 kernels SRM y trunca/normaliza.
    Input : (H, W, 3) uint8
    Output: (H, W, 15) float32 en [-1, 1]
    """
    img = img_array.astype(np.float32)
    canales = []

    for c in range(3):
        for k in KERNELS_SRM:
            r = convolve(img[:, :, c], k, mode='reflect')
            canales.append(np.clip(r, -T, T) / T)
    
    return np.stack(canales, axis=-1)  # (H, W, 15)

In [9]:
# DCT por bloques (rama espectral)
BLOCK = 8

def dct_features(img_array: np.ndarray, block_size: int = BLOCK) -> np.ndarray:
    """
    Calcula el espectro DCT log-magnitud por bloques sobre el canal Y.
    Input : (H, W, 3) uint8
    Output: (H, W, 1) float32 en [0, 1]
    """
    H, W = img_array.shape[:2]

    # RGB -> Y (luminancia BT.601) y centrado en 0
    img_f = img_array.astype(np.float32)
    Y = 0.299 * img_f[:, :, 0] + 0.587 * img_f[:, :, 1] + 0.114 * img_f[:, :, 2]
    Y -= 128.0

    dct_map = np.zeros((H, W), dtype=np.float32)
    
    for i in range(0, H - block_size + 1, block_size):
        for j in range(0, W - block_size + 1, block_size):
            bloque = Y[i:i + block_size, j:j + block_size]
            D = dctn(bloque, norm='ortho')
            dct_map[i:i + block_size, j:j + block_size] = np.log1p(np.abs(D))

    mx = dct_map.max()

    if mx > 0:
        dct_map /= mx
    
    return dct_map[:, :, np.newaxis] # (H, W, 1)

## 3. Carga de imagen con Random Crop (sin resize)

> **Importante:** redimensionar destruye el ruido de alta frecuencia que el modelo
> intenta detectar. En vez de eso, recortamos un parche aleatorio de 256×256 de la imagen original.
> Si la imagen es más chica que 256, se hace *reflect-pad* (también seguro porque no introduce ruido nuevo).


In [10]:
def cargar_imagen_random_crop(ruta: Path, size: int = CROP_SIZE,
                              rng: np.random.Generator | None = None) -> dict:
    """
    Carga una imagen y devuelve un parche aleatorio de `size × size` en dos representaciones.

    - Si la imagen es >= size en ambos lados: random crop.
    - Si es más chica: reflect-pad y luego crop centrado.
    NO usa interpolación en ningún momento.
    """
    if rng is None:
        rng = np.random.default_rng()

    img = Image.open(ruta).convert('RGB')
    arr = np.array(img, dtype=np.uint8)
    H, W, _ = arr.shape

    # Padding por reflexión si la imagen es más chica que el crop
    if H < size or W < size:
        pad_h = max(0, size - H)
        pad_w = max(0, size - W)
        arr = np.pad(
            arr,
            ((pad_h // 2, pad_h - pad_h // 2),
             (pad_w // 2, pad_w - pad_w // 2),
             (0, 0)),
            mode='reflect',
        )
        H, W, _ = arr.shape

    # Random crop
    y0 = int(rng.integers(0, H - size + 1))
    x0 = int(rng.integers(0, W - size + 1))
    parche = arr[y0:y0 + size, x0:x0 + size, :]

    return {
        'srm': calcular_residuos_srm(parche),   # (256, 256, 15)
        'dct': dct_features(parche),            # (256, 256, 1)
    }

def cargar_imagen_centro(ruta: Path, size: int = CROP_SIZE) -> dict:
    """Versión determinista (crop centrado) para validación y test."""
    img = Image.open(ruta).convert('RGB')
    arr = np.array(img, dtype=np.uint8)
    
    H, W, _ = arr.shape

    if H < size or W < size:
        pad_h = max(0, size - H)
        pad_w = max(0, size - W)
        arr = np.pad(
            arr,
            ((pad_h // 2, pad_h - pad_h // 2),
             (pad_w // 2, pad_w - pad_w // 2),
             (0, 0)),
            mode='reflect',
        )
        H, W, _ = arr.shape

    y0 = (H - size) // 2
    x0 = (W - size) // 2
    
    parche = arr[y0:y0 + size, x0:x0 + size, :]

    return {'srm': calcular_residuos_srm(parche), 'dct': dct_features(parche)}

## 4. Descubrir rutas y dividir en train / val / test

In [11]:
def collect_samples(root: Path):
    """Devuelve (lista_de_rutas_str, lista_de_labels)."""
    rutas, labels = [], []
    
    for label, folder in enumerate(CLASES):
        folder_path = root / folder

        if not folder_path.exists():
            raise FileNotFoundError(f"No se encontró: {folder_path}")
        
        for img_path in sorted(folder_path.glob("*.png")):
            rutas.append(img_path)
            labels.append(label)
    
    return rutas, labels

In [12]:
rutas_train, y_train_raw = collect_samples(ROOT_TRAIN)
rutas_val,   y_val_raw = collect_samples(ROOT_VAL)
rutas_test,  y_test_raw = collect_samples(ROOT_TEST)

In [13]:
y_train = np.array(y_train_raw, dtype = np.int32)
y_val = np.array(y_val_raw, dtype = np.int32)
y_test = np.array(y_test_raw, dtype = np.int32)

print(f"Train: {len(rutas_train)} | Val: {len(rutas_val)} | Test: {len(rutas_test)}")

Train: 32000 | Val: 8000 | Test: 9995


In [14]:
for split, (rutas, labels) in [
    ("train", (rutas_train, y_train)),
    ("val", (rutas_val, y_val)),
    ("test", (rutas_test, y_test)),
]:
    unique, counts = np.unique(labels, return_counts = True)
    print(f"\n── {split.upper()} ({len(rutas)} imágenes) ──")
    
    for u, c in zip(unique, counts):
        print(f"  {CLASES[u]:<10}: {c} imágenes")


── TRAIN (32000 imágenes) ──
  original  : 6400 imágenes
  lsb       : 6400 imágenes
  bpcs      : 6400 imágenes
  dct       : 6400 imágenes
  pvd       : 6400 imágenes

── VAL (8000 imágenes) ──
  original  : 1600 imágenes
  lsb       : 1600 imágenes
  bpcs      : 1600 imágenes
  dct       : 1600 imágenes
  pvd       : 1600 imágenes

── TEST (9995 imágenes) ──
  original  : 1999 imágenes
  lsb       : 1999 imágenes
  bpcs      : 1999 imágenes
  dct       : 1999 imágenes
  pvd       : 1999 imágenes


## 5. Pipeline `tf.data` con Random Crop

Construimos un `tf.data.Dataset` que carga imágenes de disco bajo demanda y aplica
random crop solo en entrenamiento. Esto evita meter el dataset completo en RAM
(con 256×256×15 float32 se llena rápido).


In [15]:
AUTOTUNE = tf.data.AUTOTUNE

def _make_py_loader(training: bool):
    """Devuelve una función Python que TF puede envolver con tf.py_function."""

    def _load(path_tensor, label_tensor):
        path = path_tensor.numpy().decode('utf-8')

        if training:
            res = cargar_imagen_random_crop(Path(path))
        else:
            res = cargar_imagen_centro(Path(path))
        return res['srm'].astype(np.float32), res['dct'].astype(np.float32), label_tensor
    
    return _load


def construir_dataset(rutas, labels, training: bool, batch_size: int = BATCH_SIZE):
    """tf.data.Dataset que entrega ((srm, dct), label)."""
    ds = tf.data.Dataset.from_tensor_slices((rutas, labels))

    if training:
        ds = ds.shuffle(buffer_size = len(rutas),  seed = SEED, reshuffle_each_iteration = True)

    loader = _make_py_loader(training)

    def _wrap(path, label):
        srm, dct, lab = tf.py_function(
            loader, inp = [path, label],
            Tout = [tf.float32, tf.float32, tf.int32]
        )

        srm.set_shape((CROP_SIZE, CROP_SIZE, 15))
        dct.set_shape((CROP_SIZE, CROP_SIZE, 1))
        lab.set_shape(())

        return (srm, dct), lab

    ds = ds.map(_wrap, num_parallel_calls = AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    
    return ds

In [16]:
# Convertir rutas de Path a strings para tf.data
rutas_train = [str(r) for r in rutas_train]
rutas_val = [str(r) for r in rutas_val]
rutas_test = [str(r) for r in rutas_test]

In [17]:
ds_train = construir_dataset(rutas_train, y_train, training = True)
ds_val = construir_dataset(rutas_val, y_val, training = False).cache()
ds_test = construir_dataset(rutas_test, y_test, training = False).cache()

In [18]:
for (srm_b, dct_b), lab_b in ds_train.take(1):
    print(f"SRM batch: {srm_b.shape}, DCT batch: {dct_b.shape}, labels: {lab_b.shape}")

SRM batch: (16, 256, 256, 15), DCT batch: (16, 256, 256, 1), labels: (16,)


In [19]:
for (srm_b, dct_b), lab_b in ds_train.take(1):
    print(f"SRM — shape: {srm_b.shape}, min: {srm_b.numpy().min():.3f}, max: {srm_b.numpy().max():.3f}")
    print(f"DCT — shape: {dct_b.shape}, min: {dct_b.numpy().min():.3f}, max: {dct_b.numpy().max():.3f}")
    print(f"Labels: {lab_b.numpy()}")

SRM — shape: (16, 256, 256, 15), min: -1.000, max: 1.000
DCT — shape: (16, 256, 256, 1), min: 0.000, max: 1.000
Labels: [2 1 1 4 2 3 0 0 0 3 2 4 4 4 4 0]


## 6. Class weights — combatir el desbalance

`Cover` suele tener más muestras que los algoritmos minoritarios. Sin este ajuste,
el modelo aprende a predecir `Cover` para maximizar accuracy global.


In [20]:
pesos = compute_class_weight(
    class_weight = 'balanced',
    classes = np.arange(N_CLASSES),
    y = y_train
)

class_weight = {i: float(w) for i, w in enumerate(pesos)}

print("Class weights:")

for i, w in class_weight.items():
    print(f"  {NOMBRES_CLASES[i]:<10}: {w:.3f}")

Class weights:
  original  : 1.000
  lsb       : 1.000
  bpcs      : 1.000
  dct       : 1.000
  pvd       : 1.000


## 7. Bloques de la arquitectura

### Mish
Activación suave (`x * tanh(softplus(x))`), funciona mejor que ReLU en problemas de
ruido sutil porque no satura en 0 y tiene gradiente no nulo en valores negativos pequeños.

### Bloque residual
`Conv → BN → Mish → Conv → BN` + atajo `1×1` cuando cambia el número de filtros.

### Squeeze-and-Excitation (SE)
Aprende un peso por canal: `GAP → Dense(C/r, ReLU) → Dense(C, Sigmoid) → multiplica`.
Aquí lo usamos sobre el vector concatenado de las dos ramas, para que la red
decida cuánto pesar SRM vs DCT por instancia.


In [21]:
def mish(x):
    """Activación Mish: x * tanh(softplus(x))."""
    return x * tf.math.tanh(tf.math.softplus(x))

def conv_bn_mish(x, filtros, kernel = 3, stride = 1, name = None):
    """Conv2D + BatchNorm + Mish."""
    x = layers.Conv2D(
        filtros, kernel, strides = stride, padding = 'same',
        use_bias = False,
        kernel_regularizer = regularizers.l2(L2_REG),
        name = f"{name}_conv" if name else None,
    )(x)

    x = layers.BatchNormalization(name = f"{name}_bn" if name else None)(x)
    x = layers.Activation(mish, name = f"{name}_mish" if name else None)(x)
    
    return x

def bloque_residual(x, filtros, stride = 1, name = "res"):
    """
    Bloque residual estilo ResNet:
        entrada → Conv-BN-Mish → Conv-BN → (+ atajo) → Mish
    """
    atajo = x

    if stride != 1 or x.shape[-1] != filtros:
        atajo = layers.Conv2D(
            filtros, 1, strides = stride, padding = 'same', use_bias = False,
            kernel_regularizer = regularizers.l2(L2_REG),
            name = f"{name}_short_conv",
        )(atajo)
        atajo = layers.BatchNormalization(name = f"{name}_short_bn")(atajo)

    x = conv_bn_mish(x, filtros, 3, stride, name = f"{name}_a")
    x = layers.Conv2D(
        filtros, 3, padding = 'same', use_bias = False,
        kernel_regularizer = regularizers.l2(L2_REG),
        name = f"{name}_b_conv",
    )(x)
    
    x = layers.BatchNormalization(name = f"{name}_b_bn")(x)
    x = layers.Add(name = f"{name}_add")([x, atajo])
    x = layers.Activation(mish, name = f"{name}_out")(x)

    return x

def bloque_se(x, ratio: int = 8, name: str = "se"):
    """
    Squeeze-and-Excitation sobre un tensor 1-D (post-GAP / post-concat).
    Aprende un peso por dimensión y lo multiplica de vuelta.

    Aquí 'x' viene como (batch, C) tras el GAP+Concat. Lo expandimos a
    (batch, 1, C) para reutilizar Dense con activación canal a canal.
    """
    C = x.shape[-1]
    r = max(C // ratio, 4)

    s = layers.Dense(r, activation = 'relu',
                     kernel_regularizer = regularizers.l2(L2_REG),
                     name = f"{name}_squeeze")(x)
    
    s = layers.Dense(C, activation = 'sigmoid',
                     kernel_regularizer = regularizers.l2(L2_REG),
                     name = f"{name}_excite")(s)
    
    return layers.Multiply(name = f"{name}_scale")([x, s])

## 8. Definición de las ramas y el modelo completo

In [22]:
def rama_srm(entrada):
    """Rama espacial: procesa los 15 mapas de residuos SRM."""
    x = conv_bn_mish(entrada, 32, kernel = 3, name = "srm_stem")          # 256
    x = bloque_residual(x, 32,  stride = 1, name = "srm_b1")
    x = layers.AveragePooling2D(2, name = "srm_pool1")(x)               # 128

    x = bloque_residual(x, 64,  stride = 1, name = "srm_b2")
    x = layers.AveragePooling2D(2, name = "srm_pool2")(x)               # 64

    x = bloque_residual(x, 96,  stride = 1, name = "srm_b3")
    x = layers.AveragePooling2D(2, name = "srm_pool3")(x)               # 32

    x = bloque_residual(x, 128, stride = 1, name = "srm_b4")
    x = layers.AveragePooling2D(2, name = "srm_pool4")(x)               # 16

    x = bloque_residual(x, 128, stride = 1, name = "srm_b5")
    x = layers.GlobalAveragePooling2D(name = "srm_gap")(x)              # (B, 128)
    
    return x

def rama_dct(entrada):
    """Rama espectral: procesa el espectro DCT (1 canal). Usa SeparableConv para ser ligera."""
    x = layers.Conv2D(16, 3, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2_REG),
                      name = "dct_stem_conv")(entrada)
    x = layers.BatchNormalization(name = "dct_stem_bn")(x)
    x = layers.Activation(mish, name = "dct_stem_mish")(x)

    # SeparableConv para extraer patrones del espectro de forma eficiente
    def sep_block(x, f, name):
        x = layers.SeparableConv2D(
            f, 3, padding = 'same', use_bias=False,
            depthwise_regularizer = regularizers.l2(L2_REG),
            pointwise_regularizer = regularizers.l2(L2_REG),
            name=f"{name}_sep",
        )(x)

        x = layers.BatchNormalization(name = f"{name}_bn")(x)
        x = layers.Activation(mish, name = f"{name}_mish")(x)

        return x

    x = sep_block(x, 32,  "dct_s1"); x = layers.AveragePooling2D(2, name = "dct_p1")(x)   # 128
    x = sep_block(x, 64,  "dct_s2"); x = layers.AveragePooling2D(2, name = "dct_p2")(x)   # 64
    x = sep_block(x, 96,  "dct_s3"); x = layers.AveragePooling2D(2, name = "dct_p3")(x)   # 32
    x = sep_block(x, 128, "dct_s4"); x = layers.AveragePooling2D(2, name = "dct_p4")(x)   # 16
    x = sep_block(x, 128, "dct_s5")
    x = layers.GlobalAveragePooling2D(name = "dct_gap")(x)              # (B, 128)
    return x

def construir_modelo(n_clases: int = N_CLASSES) -> tf.keras.Model:
    """
    Modelo completo:
        SRM input → rama_srm → GAP ─┐
                                    ├─ Concat ─→ SE ─→ Dense+Dropout ─→ softmax
        DCT input → rama_dct → GAP ─┘
    """
    in_srm = layers.Input(shape = (CROP_SIZE, CROP_SIZE, 15), name="input_srm")
    in_dct = layers.Input(shape = (CROP_SIZE, CROP_SIZE,  1), name="input_dct")

    f_srm = rama_srm(in_srm)        # (B, 128)
    f_dct = rama_dct(in_dct)        # (B, 128)

    # Fusión tardía
    fused = layers.Concatenate(name = "late_fusion")([f_srm, f_dct])     # (B, 256)

    # Atención SE sobre el vector fusionado
    fused = bloque_se(fused, ratio = 8, name = "se_fusion")                # (B, 256)

    # Cabezal denso fuertemente regularizado
    x = layers.Dense(
        128, kernel_regularizer = regularizers.l2(L2_REG),
        name = "head_dense_1"
    )(fused)
    x = layers.BatchNormalization(name = "head_bn_1")(x)
    x = layers.Activation(mish, name = "head_mish_1")(x)
    x = layers.Dropout(DROPOUT, name = "head_drop_1")(x)

    x = layers.Dense(
        64, kernel_regularizer = regularizers.l2(L2_REG),
        name = "head_dense_2"
    )(x)
    x = layers.BatchNormalization(name = "head_bn_2")(x)
    x = layers.Activation(mish, name = "head_mish_2")(x)
    x = layers.Dropout(DROPOUT, name = "head_drop_2")(x)

    salida = layers.Dense(
        n_clases, activation = 'softmax',
        kernel_regularizer = regularizers.l2(L2_REG),
        name = "softmax_out"
    )(x)

    return models.Model(inputs = [in_srm, in_dct], outputs = salida, name = "StegoCNN_v4")

modelo = construir_modelo()
modelo.summary(line_length = 110)


Model: "StegoCNN_v4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                   ┃ Output Shape              ┃          Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_srm (InputLayer)         │ (None, 256, 256, 15)      │                0 │ -                          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_stem_conv (Conv2D)         │ (None, 256, 256, 32)      │            4,320 │ input_srm[0][0]            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_stem_bn                    │ (None, 256, 256, 32)      │              128 │ srm_stem_conv[0][0]        │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_stem_mish (Activation)     │ (None, 256, 256, 32)      │                0 │ srm_stem_bn[0][0]          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_a_conv (Conv2D)         │ (None, 256, 256, 32)      │            9,216 │ srm_stem_mish[0][0]        │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_a_bn                    │ (None, 256, 256, 32)      │              128 │ srm_b1_a_conv[0][0]        │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_a_mish (Activation)     │ (None, 256, 256, 32)      │                0 │ srm_b1_a_bn[0][0]          │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_b_conv (Conv2D)         │ (None, 256, 256, 32)      │            9,216 │ srm_b1_a_mish[0][0]        │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_b_bn                    │ (None, 256, 256, 32)      │              128 │ srm_b1_b_conv[0][0]        │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_add (Add)               │ (None, 256, 256, 32)      │                0 │ srm_b1_b_bn[0][0],         │
│                                │                           │                  │ srm_stem_mish[0][0]        │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b1_out (Activation)        │ (None, 256, 256, 32)      │                0 │ srm_b1_add[0][0]           │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_pool1 (AveragePooling2D)   │ (None, 128, 128, 32)      │                0 │ srm_b1_out[0][0]           │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b2_a_conv (Conv2D)         │ (None, 128, 128, 64)      │           18,432 │ srm_pool1[0][0]            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ srm_b2_a_bn                    │ (None, 128, 128, 64)      │              256 │ srm_b2_a_conv[0][0]        │
│ (BatchNormalization)           │                           │                  │                            │
├────────────────────────────────┼───────────────────────────┼──────────────────┼────────────────────────────┤
│ sr

 Total params: 895,909 (3.42 MB)

 Trainable params: 892,165 (3.40 MB)

 Non-trainable params: 3,744 (14.62 KB)

In [23]:
(srm_b, dct_b), lab_b = next(iter(ds_train))
out = modelo((srm_b, dct_b), training=False)

print(f"Output shape: {out.shape}")
print(f"Suma de probs (debe ser ~1.0): {out.numpy().sum(axis = 1)[:5]}")

Output shape: (16, 5)
Suma de probs (debe ser ~1.0): [1.         1.         0.99999994 1.         1.        ]


## 9. Compilación — AdamW + Cosine Decay

`AdamW` desacopla el weight decay del gradiente (a diferencia del L2 dentro de Adam),
lo cual generaliza mucho mejor en problemas de ruido sutil como esteganálisis.


In [24]:
# Verificar GPU disponible
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs disponibles: {gpus}")

# Permitir crecimiento de memoria en lugar de reservar todo de golpe
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth activado — TF usará GPU solo cuando la necesite")

GPUs disponibles: []


In [25]:
# Pasos por época (aproximado, depende del último batch)
pasos_por_epoca = math.ceil(len(rutas_train) / BATCH_SIZE)
total_pasos = pasos_por_epoca * EPOCHS
# 5 épocas de warmup
warmup_pasos  = pasos_por_epoca * 5
decay_pasos   = total_pasos - warmup_pasos


# Cosine decay: LR baja suavemente de LR_INIT a casi 0 a lo largo del entrenamiento
lr_schedule = optimizers.schedules.CosineDecay(
    initial_learning_rate = LR_INIT,
    decay_steps = decay_pasos,
    alpha = 1e-2,
    warmup_target = LR_INIT,
    warmup_steps = warmup_pasos,
)

# AdamW está en tf.keras.optimizers desde TF 2.11
optimizador = optimizers.AdamW(
    learning_rate = lr_schedule,
    weight_decay = WEIGHT_DECAY,
    beta_1 = 0.9, beta_2 = 0.999, epsilon = 1e-7,
)

modelo.compile(
    optimizer = optimizador,
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = False),
    metrics = [
        tf.keras.metrics.SparseCategoricalAccuracy(name = 'acc'),
    ],
)
print("Modelo compilado con AdamW + CosineDecay.")

Modelo compilado con AdamW + CosineDecay.


## 10. Callbacks

Calculamos **Macro F1** sobre val al final de cada época y lo usamos como métrica
para `EarlyStopping` y `ModelCheckpoint` — más confiable que `val_acc` en multiclase desbalanceado.


In [ ]:
class MacroF1Callback(callbacks.Callback):
    """Calcula Macro F1 sobre el dataset de validación al final de cada época."""
    def __init__(self, val_ds, n_clases=N_CLASSES):
        super().__init__()
        self.val_ds = val_ds
        self.n_clases = n_clases
        self.best_f1  = 0.0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        y_true, y_pred = [], []
        
        for (srm_b, dct_b), lab_b in self.val_ds:
            probs = self.model.predict_on_batch((srm_b, dct_b))
            y_true.append(lab_b.numpy())
            y_pred.append(np.argmax(probs, axis=1))
        
        y_true = np.concatenate(y_true)
        y_pred = np.concatenate(y_pred)
        
        macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
        
        logs['val_macro_f1'] = float(macro)
        
        # Indicador visual de mejora
        mejora = " ^ mejor" if macro > self.best_f1 else ""
        self.best_f1 = max(self.best_f1, macro)
        print(f" — val_macro_f1: {macro:.4f}{mejora}")


cbs = [
    MacroF1Callback(ds_val),

    callbacks.ModelCheckpoint(
        'stego_cnn_v4_best.keras',
        monitor='val_macro_f1', mode = 'max',
        save_best_only = True, verbose = 1,
    ),
    
    # Para temprano si val_macro_f1 no mejora en 15 épocas
    callbacks.EarlyStopping(
        monitor = 'val_macro_f1', mode='max',
        patience = 15,
        restore_best_weights = True,
        verbose = 1,
    ),

    # Reduce el learning rate si val_acc se estanca
    callbacks.ReduceLROnPlateau(
        monitor='val_acc', mode='max',
        factor=0.5, # divide LR a la mitad
        patience=5, # si en 5 épocas no mejora val_acc
        min_lr=1e-6,
        verbose=1,
    ),

    callbacks.CSVLogger('stego_cnn_v4_log.csv'),

    # Imprime el learning rate actual
    callbacks.LambdaCallback(
        on_epoch_begin=lambda epoch, logs: print(
            f"\nÉpoca {epoch+1} — LR actual: "
            f"{float(tf.keras.backend.get_value(modelo.optimizer.learning_rate)):.2e}"
        )
    ),
]

: 

## 11. Entrenamiento

In [ ]:
history = modelo.fit(
    ds_train,
    validation_data = ds_val,
    epochs = EPOCHS,
    class_weight = class_weight,
    callbacks = cbs,
    verbose = 1,
)


Época 1 — LR actual: 5.00e-04
Epoch 1/80
2000/2000 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - acc: 0.7104 - loss: 0.8935

## 12. Curvas de entrenamiento

In [1]:
fig, axes = plt.subplots(1, 3, figsize = (18, 4))

axes[0].plot(history.history['loss'], label = 'train')
axes[0].plot(history.history['val_loss'], label = 'val')
axes[0].set_title('Loss'); axes[0].set_xlabel('época'); axes[0].legend(); axes[0].grid(alpha = 0.3)

axes[1].plot(history.history['acc'], label = 'train')
axes[1].plot(history.history['val_acc'], label = 'val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('época'); axes[1].legend(); axes[1].grid(alpha = 0.3)

if 'val_macro_f1' in history.history:
    axes[2].plot(history.history['val_macro_f1'], label = 'val', color = 'C2')
    axes[2].set_title('Macro F1 (val)'); axes[2].set_xlabel('época'); axes[2].legend(); axes[2].grid(alpha = 0.3)

plt.tight_layout()
plt.savefig('curvas_entrenamiento_v4.png', dpi = 120)
plt.show()

NameError: name 'plt' is not defined

## 13. Evaluación final en test

Métrica principal: **Macro F1**. Métrica secundaria: matriz de confusión normalizada por fila
(porcentaje de aciertos por clase real). La diagonal te dice qué tan bien se distingue cada algoritmo.


In [ ]:
y_true_test, y_pred_test = [], []

for (srm_b, dct_b), lab_b in ds_test:
    probs = modelo.predict_on_batch((srm_b, dct_b))
    y_true_test.append(lab_b.numpy())
    y_pred_test.append(np.argmax(probs, axis = 1))

y_true_test = np.concatenate(y_true_test)
y_pred_test = np.concatenate(y_pred_test)

print("── Classification report ──")
print(classification_report(
    y_true_test, y_pred_test,
    target_names=NOMBRES_CLASES, digits = 4, zero_division = 0,
))

macro = f1_score(y_true_test, y_pred_test, average = 'macro', zero_division = 0)
print(f"\nMacro F1 (test): {macro:.4f}")

In [ ]:
# Matriz de confusión normalizada por fila
cm = confusion_matrix(y_true_test, y_pred_test, labels = list(range(N_CLASSES)))
cm_norm = cm.astype(np.float32) / cm.sum(axis = 1, keepdims = True).clip(min = 1)

fig, axes = plt.subplots(1, 2, figsize = (14, 5))

sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues',
            xticklabels = NOMBRES_CLASES, yticklabels = NOMBRES_CLASES, ax = axes[0])
axes[0].set_title('Matriz de confusión (cuentas)')
axes[0].set_xlabel('predicho'); axes[0].set_ylabel('real')

sns.heatmap(cm_norm, annot = True, fmt = '.2f', cmap = 'Blues', vmin = 0, vmax = 1,
            xticklabels = NOMBRES_CLASES, yticklabels = NOMBRES_CLASES, ax = axes[1])
axes[1].set_title('Matriz de confusión (% por fila)')
axes[1].set_xlabel('predicho'); axes[1].set_ylabel('real')

plt.tight_layout()
plt.savefig('confusion_matrix_v4.png', dpi = 120)
plt.show()

## 14. (Opcional) Inspeccionar la atención SE

¿La red está pesando más SRM o DCT? Los primeros 128 pesos del vector excite corresponden
a la rama SRM y los 128 siguientes a la DCT. Comparamos sus medias.


In [ ]:
# Modelo auxiliar que expone la salida de la capa SE
se_layer = modelo.get_layer('se_fusion_scale')
modelo_inspect = tf.keras.Model(modelo.inputs, se_layer.output)

pesos_srm, pesos_dct, etiquetas = [], [], []
for (srm_b, dct_b), lab_b in ds_test.take(20):  # primeras 20 batches
    out = modelo_inspect.predict_on_batch((srm_b, dct_b))   # (B, 256)
    # out ya está escalado (x * s). Para ver el peso "puro" hacemos out / x_original
    # — pero por simplicidad reportamos la magnitud media de cada mitad.
    pesos_srm.append(np.abs(out[:, :128]).mean(axis=1))
    pesos_dct.append(np.abs(out[:, 128:]).mean(axis=1))
    etiquetas.append(lab_b.numpy())

pesos_srm = np.concatenate(pesos_srm)
pesos_dct = np.concatenate(pesos_dct)
etiquetas = np.concatenate(etiquetas)

print("Activación media por rama y clase (post-SE):")
print(f"{'Clase':<10} {'|SRM|':>8} {'|DCT|':>8} {'ratio DCT/SRM':>16}")

for c in range(N_CLASSES):
    mask = etiquetas == c

    if mask.sum() == 0:
        continue
    
    s = pesos_srm[mask].mean()
    d = pesos_dct[mask].mean()

    print(f"{NOMBRES_CLASES[c]:<10} {s:>8.4f} {d:>8.4f} {d / max(s, 1e-9):>16.3f}")

## 15. Guardar el modelo final

In [ ]:
modelo.save('stego_cnn_v4_final.keras')

In [ ]:
print("Modelo guardado en stego_cnn_v4_final.keras")
print("Mejor checkpoint (por val_macro_f1) en stego_cnn_v4_best.keras")

---

### Resumen de decisiones clave

| Aspecto | Decisión |
|---|---|
| Tamaño de entrada | **Random Crop 256×256** (sin resize ni recompresión) |
| Rama espacial | 15 canales SRM, bloques residuales con Mish |
| Rama espectral | DCT 8×8 sobre canal Y, SeparableConv para eficiencia |
| Fusión | Late fusion: `GAP → Concat → SE` |
| Atención | Squeeze-and-Excitation post-concat (decide SRM vs DCT) |
| Regularización | L2 = 1e-4 en todas las conv y dense, Dropout = 0.45 |
| Optimizador | **AdamW** + `CosineDecay` |
| Desbalance | `class_weight='balanced'` |
| Métrica principal | **Macro F1** (no accuracy) + matriz de confusión normalizada |

### Si el modelo se sobreajusta
- Sube `DROPOUT` a 0.55, `L2_REG` a 5e-4.
- Reduce los filtros máximos de 128 → 96.
- Aumenta `BATCH_SIZE` si la GPU lo permite (estabiliza BN).

### Si el modelo no aprende
- Verifica que el payload de tus algoritmos sea ≥25% (sin esto no hay señal).
- Asegúrate de que las imágenes **nunca** pasaron por JPEG después de aplicar el stego.
- Imprime los `class_weight`: si una clase tiene peso >5, te falta data de esa clase.
